**No SENTINEL**

In [ ]:
# Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
def parse_conll_file(conll_file_path):
    data = []
    tokens = []
    ner_tags = []

    # Unified tag mapping (B- and I- merged)
    tag2id = {
        "O": 0,
        "Implicit": 1,
        "Explicit": 2
    }

    with open(conll_file_path, 'r') as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if line == "":
                if tokens:
                    data.append({
                        'id': str(len(data)),
                        'ner_tags': ner_tags,
                        'tokens': tokens
                    })
                    tokens = []
                    ner_tags = []
            else:
                parts = line.split()
                if len(parts) != 3:
                    print(f"Skipping malformed line {i}: {line}")
                    continue

                token, _, tag = parts
                tokens.append(token)

                # Merge B- and I- tags
                if tag in ["B-Implicit", "I-Implicit"]:
                    ner_tags.append(tag2id["Implicit"])
                elif tag in ["B-Explicit", "I-Explicit"]:
                    ner_tags.append(tag2id["Explicit"])
                elif tag == "O":
                    ner_tags.append(tag2id["O"])
                else:
                    print(f"Unknown tag '{tag}' found, skipping line: {line}")
                    continue

        if tokens:
            data.append({
                'id': str(len(data)),
                'ner_tags': ner_tags,
                'tokens': tokens
            })

    return data

In [ ]:
# Example usage
conll_file_path = "/path/to/your/file"
data = parse_conll_file(conll_file_path)

# Print the formatted data
for entry in data:
    print(entry)

In [ ]:
print(len(data))

In [ ]:
!pip install transformers
!pip install accelerate -U
!pip install datasets

In [ ]:
!pip install bitsandbytes accelerate transformers

In [ ]:
# Mapping for conversion
id2label = {0: "O", 1: "Implicit", 2: "Explicit"}
label2id = {"O": 0, "Implicit": 1, "Explicit": 2}

In [ ]:
# A function to attach punctuation to sentences
import re

def reconstruct_sentence(tokens):
    sentence = ""
    for i, token in enumerate(tokens):
        if re.match(r"^[.,!?;:’”')\]]$", token):
            sentence += token
        elif token in ['(', '[', '“', '‘']:
            if sentence and not sentence.endswith(" "):
                sentence += " "
            sentence += token
        else:
            if sentence and not sentence.endswith(" "):
                sentence += " "
            sentence += token
    return sentence

In [ ]:
# Creating a df with sentences instead of separate tokens
import pandas as pd

records = []
for sent in data:
    sentence_txt = reconstruct_sentence(sent["tokens"])
    label_txt = " ".join(id2label[t] for t in sent["ner_tags"])
    records.append({"sentence": sentence_txt, "ner_tag": label_txt})

df = pd.DataFrame(records)

In [ ]:
print(df.shape)

In [ ]:
print(df.head(5))

In [ ]:
from huggingface_hub import login

# Make sure you are logged in to Hugging Face
# You can authenticate by running: huggingface-cli login
# Or by passing your token: login(token="YOUR_TOKEN")
login()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [ ]:
def build_prompt(sentence):
    prompt = f"""Your task is to analyze each sentence of the text and determine whether it is *Implicit* or *Explicit*.
Explicit refers to transparent and clearly understandable content.
Implicit refers to hidden meanings or assumptions that are unclear from the given text alone.
The text surrounding the sentence is your context.

Instructions:
- For each Implicit or Explicit sentence found, return a separate JSON object with exactly two fields:
  - "sentence": the exact span from the input sentence expressing the argument.
  - "type": "Explicit" or "Implicit".
- If none of them is found, return one JSON object with both fields set to "".
- Do **not** wrap the JSON objects in a list (no square brackets).
- Separate multiple JSON objects with commas and spaces only, e.g.:
  {{ "sentence": "...", "type": "Implicit" }}, {{ "sentence": "...", "type": "Explicit" }}
- The output must be strictly valid JSON:
  - Use double quotes only
  - Close all braces correctly
  - Do not include trailing commas
- Do not include any explanation, notes, or extra text. Output only the JSON objects.

Sentence:
{sentence}
"""
    return prompt

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb_config)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=2048, do_sample=False)

In [ ]:
predictions = []

for row in tqdm(df.itertuples(), total=len(df)):
    prompt = build_prompt(row.sentence)
    output = pipe(prompt)[0]['generated_text']

    output_clean = output.replace(prompt, "").strip()

    predictions.append({
        "sentence": row.sentence,
        "gold": row.ner_tag,
        "mistral_output": output_clean
    })

In [ ]:
import json

In [ ]:
output_path = "/path/to/your/project/binary_predictions_2048.json"
with open(output_path, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"Saved predictions to {output_path}")

In [ ]:
with open("/path/to/your/project/binary_predictions_2048.json") as f:
    predictions = json.load(f)

In [ ]:
print(predictions[1]["sentence"])
print(predictions[1]["gold"])
print(predictions[1]["mistral_output"])

In [ ]:
import json
import nltk
from difflib import SequenceMatcher
from sklearn.metrics import classification_report
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt_tab')

In [ ]:
import re

def extract_mistral_sentences(mistral_output):
    mistral_output = mistral_output.strip()

    # Remove the "Output:" prefix if present
    if mistral_output.lower().startswith("output:"):
        mistral_output = mistral_output[len("output:"):].strip()

    # Extract JSON-like pairs: {"sentence": "...", "type": "..."}
    pattern = r'\{\s*"sentence"\s*:\s*"(.+?)"\s*,\s*"type"\s*:\s*"(.+?)"\s*\}'
    matches = re.findall(pattern, mistral_output, re.DOTALL)

    return [(s.strip(), t.strip().lower()) for s, t in matches]

def get_sentence_level_labels(text, gold_labels):
    tokens = word_tokenize(text.strip())
    labels = gold_labels.strip().split()

    if len(tokens) != len(labels):
        print(f"⚠️ Warning: token-label mismatch: {len(tokens)} tokens vs {len(labels)} labels")

    sentences = sent_tokenize(text.strip())
    sentence_labels = []

    idx = 0
    for sent in sentences:
        sent_len = len(word_tokenize(sent))
        sent_labels = labels[idx:idx + sent_len]

        if not sent_labels:
            sentence_labels.append((sent.strip(), "O"))
        elif any("Implicit" in tag for tag in sent_labels):
            sentence_label = "Implicit"
        elif any("Explicit" in tag for tag in sent_labels):
            sentence_label = "Explicit"
        else:
            sentence_label = "O"

        sentence_labels.append((sent.strip(), sentence_label))
        idx += sent_len

    return sentence_labels

def best_match(pred_sent, gold_sents):
    pred_clean = pred_sent.lower()
    best_idx, best_score = -1, 0
    for i, (gold_sent, _) in enumerate(gold_sents):
        score = SequenceMatcher(None, pred_clean, gold_sent.lower()).ratio()
        if score > best_score:
            best_idx, best_score = i, score
    return best_idx if best_score > 0.8 else -1  # threshold

gold_all = []
pred_all = []

for item in predictions:
    gold_pairs = get_sentence_level_labels(item["sentence"], item["gold"])
    pred_pairs = extract_mistral_sentences(item["mistral_output"])

    matched = set()
    pred_sent_map = {}

    for pred_sent, pred_label in pred_pairs:
        idx = best_match(pred_sent, gold_pairs)
        if idx != -1 and idx not in matched:
            pred_sent_map[idx] = pred_label
            matched.add(idx)

    for i, (gold_sent, gold_label) in enumerate(gold_pairs):
      pred_label = pred_sent_map.get(i, "O")  # assign "O" if not predicted

      # Normalize both labels to have capitalized first letter
      gold_label = gold_label.strip().capitalize()
      pred_label = pred_label.strip().capitalize()

      gold_all.append(gold_label)
      pred_all.append(pred_label)

print(classification_report(
    gold_all,
    pred_all,
    labels=["Implicit", "Explicit", "O"],
    digits=4
))

___________________________
___________________________
___________________________

**SENTINEL**

In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def parse_conll_file(conll_file_path):
    data = []
    tokens = []
    ner_tags = []

    tag2id = {
        "O": 0,
        "B-Implicit": 1,
        "I-Implicit": 2,
        "B-Explicit": 3,
        "I-Explicit": 4
    }

    with open(conll_file_path, 'r') as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if line == "":
                if tokens:
                    data.append({
                        'id': str(len(data)),
                        'ner_tags': ner_tags,
                        'tokens': tokens
                    })
                    tokens = []
                    ner_tags = []
            else:
                parts = line.split()
                if len(parts) != 3:
                    print(f"Skipping malformed line {i}: {line}")
                    continue

                token, _, tag = parts
                tokens.append(token)

                if tag in tag2id:
                    ner_tags.append(tag2id[tag])
                else:
                    print(f"Unknown tag '{tag}' found, skipping line: {line}")
                    continue

        if tokens:
            data.append({
                'id': str(len(data)),
                'ner_tags': ner_tags,
                'tokens': tokens
            })

    return data

In [ ]:
# Example usage
conll_file_path = "/path/to/your/file"
data = parse_conll_file(conll_file_path)

# Print the formatted data
for entry in data:
    print(entry)

In [ ]:
!pip install transformers
!pip install --upgrade "transformers>=4.40"
!pip install accelerate -U
!pip install datasets

In [ ]:
!pip install bitsandbytes accelerate transformers

In [ ]:
# Mapping for conversion
id2label = {0: "O", 1: "B-Implicit", 2: "I-Implicit", 3: "B-Explicit", 4: "I-Explicit"}
label2id = {"O": 0, "B-Implicit": 1, "I-Implicit": 2, "B-Explicit": 3, "I-Explicit": 4}

In [ ]:
# A function to attach punctuation to sentences
import re

def reconstruct_sentence(tokens):
    sentence = ""
    for i, token in enumerate(tokens):
        if re.match(r"^[.,!?;:’”')\]]$", token):
            sentence += token
        elif token in ['(', '[', '“', '‘']:
            if sentence and not sentence.endswith(" "):
                sentence += " "
            sentence += token
        else:
            if sentence and not sentence.endswith(" "):
                sentence += " "
            sentence += token
    return sentence

In [ ]:
# Creating a df with sentences instead of separate tokens
import pandas as pd

records = []
for sent in data:
    sentence_txt = reconstruct_sentence(sent["tokens"])
    label_txt = " ".join(id2label[t] for t in sent["ner_tags"])
    records.append({"sentence": sentence_txt, "ner_tag": label_txt})

df = pd.DataFrame(records)

In [ ]:
print(df.shape)

In [ ]:
print(df.head(5))

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
    pipeline,
)
import torch

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
pad_id = tokenizer.pad_token_id

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)
model.resize_token_embeddings(len(tokenizer))

In [ ]:
SENTINEL = "### END"

def build_prompt(sentence):
    prompt = f"""Your task is to analyze the argumentative sentence and determine whether it is *Implicit* or *Explicit*.
Explicit refers to transparent and clearly understandable content.
Implicit refers to a hidden assumption, missing link, or implied reasoning not directly stated.
Be especially careful to detect Implicit.

Instructions:
- For each Implicit or Explicit sentence found, return a separate JSON object with exactly two fields:
  - "sentence": the exact span from the input sentence expressing the argument.
  - "type": "Explicit" or "Implicit".
- If none of them is found, return one JSON object with both fields set to "".
- Return multiple JSON objects separated by commas and spaces.
- Output valid JSON only — use double quotes, no trailing commas, and close all braces.
- Do NOT include explanations or extra text.
- End your response with: {SENTINEL}

Sentence:
{sentence}
"""
    return prompt

In [ ]:
gen_cfg = GenerationConfig(
    # length
    max_new_tokens=1024,
    early_stopping=False,
    pad_token_id=pad_id,
    eos_token_id=tokenizer.eos_token_id,
    do_sample=False,
    temperature=0.1,
    top_p=0.95,
)

model.generation_config = gen_cfg

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    return_full_text=False,
)

In [ ]:
predictions = []

for row in tqdm(df.itertuples(), total=len(df)):
    prompt = build_prompt(row.sentence)
    output = pipe(prompt, generation_config=gen_cfg)[0]['generated_text']

    output_clean = output.strip()
    output_clean = output_clean.split(SENTINEL)[0].strip()

    predictions.append({
        "sentence": row.sentence,
        "gold": row.ner_tag,
        "mistral_output": output_clean
    })

In [ ]:
import json

In [ ]:
output_path = "/path/to/your/project/SENTINEL_binary_predictions.json"
with open(output_path, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"Saved predictions to {output_path}")

In [ ]:
with open("/path/to/your/project/SENTINEL_binary_predictions.json") as f:
    predictions = json.load(f)

In [ ]:
import nltk
from difflib import SequenceMatcher
from sklearn.metrics import classification_report
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt_tab')

In [ ]:
import re

def extract_mistral_sentences(mistral_output):
    mistral_output = mistral_output.strip()

    # Remove the "Output:" prefix if present
    if mistral_output.lower().startswith("output:"):
        mistral_output = mistral_output[len("output:"):].strip()

    # Extract JSON-like pairs: {"sentence": "...", "type": "..."}
    pattern = r'\{\s*"sentence"\s*:\s*"(.+?)"\s*,\s*"type"\s*:\s*"(.+?)"\s*\}'
    matches = re.findall(pattern, mistral_output, re.DOTALL)

    return [(s.strip(), t.strip().lower()) for s, t in matches]

def get_sentence_level_labels(text, gold_labels):
    tokens = word_tokenize(text.strip())
    labels = gold_labels.strip().split()

    if len(tokens) != len(labels):
        print(f"⚠️ Warning: token-label mismatch: {len(tokens)} tokens vs {len(labels)} labels")

    sentences = sent_tokenize(text.strip())
    sentence_labels = []

    idx = 0
    for sent in sentences:
        sent_len = len(word_tokenize(sent))
        sent_labels = labels[idx:idx + sent_len]

        if not sent_labels:
            sentence_labels.append((sent.strip(), "O"))
        elif any("Implicit" in tag for tag in sent_labels):
            sentence_label = "Implicit"
        elif any("Explicit" in tag for tag in sent_labels):
            sentence_label = "Explicit"
        else:
            sentence_label = "O"

        sentence_labels.append((sent.strip(), sentence_label))
        idx += sent_len

    return sentence_labels

def best_match(pred_sent, gold_sents):
    pred_clean = pred_sent.lower()
    best_idx, best_score = -1, 0
    for i, (gold_sent, _) in enumerate(gold_sents):
        score = SequenceMatcher(None, pred_clean, gold_sent.lower()).ratio()
        if score > best_score:
            best_idx, best_score = i, score
    return best_idx if best_score > 0.8 else -1  # threshold

gold_all = []
pred_all = []

for item in predictions:
    gold_pairs = get_sentence_level_labels(item["sentence"], item["gold"])
    pred_pairs = extract_mistral_sentences(item["mistral_output"])

    matched = set()
    pred_sent_map = {}

    for pred_sent, pred_label in pred_pairs:
        idx = best_match(pred_sent, gold_pairs)
        if idx != -1 and idx not in matched:
            pred_sent_map[idx] = pred_label
            matched.add(idx)

    for i, (gold_sent, gold_label) in enumerate(gold_pairs):
      pred_label = pred_sent_map.get(i, "O")  # assign "O" if not predicted

      # Normalize both labels to have capitalized first letter
      gold_label = gold_label.strip().capitalize()
      pred_label = pred_label.strip().capitalize()

      gold_all.append(gold_label)
      pred_all.append(pred_label)

print(classification_report(
    gold_all,
    pred_all,
    labels=["Implicit", "Explicit"],
    digits=4
))